# LensIQ: deploy a Roboflow Universe detector as a Databricks-native endpoint

Registers an MLflow PyFunc that wraps `inference.get_model(...)` to
Unity Catalog and serves it behind its own Databricks Model Serving
endpoint. The PyFunc downloads weights on first load via the official
Roboflow Inference SDK (the only download path that works for any
Universe model, regardless of whether the owner enabled `.pt` exports).
Per-frame inference runs **locally inside the endpoint** via
`self._model.infer(...)`.

This is the same lifecycle the `dbx-models/spill_detect_model.py`
pattern uses: deploy is fast (no big artifact), cold start downloads
weights once into the serving container's writable temp dir, and every
subsequent frame stays on Databricks.

Re-run this notebook once per use case (license plate, spill, wet
floor sign, slip & fall) with different widgets - the
bundle's `lensiq_deploy_roboflow_detectors` job wires that up as a
multi-task fan-out.

Why one endpoint per use case (instead of one PyFunc dispatching by
`model_id`):

- Independent UC registry entry per model -> per-use-case versioning + RBAC.
- Independent serving config (workload size, scale-to-zero, alerts).
- Single-purpose endpoints make ownership / cost-attribution obvious.
- One endpoint can be updated without redeploying the others.

Cold-start behavior:

The first request after a scale-from-zero downloads weights from
Roboflow into `MODEL_CACHE_DIR` (a writable temp dir), then runs
inference. Subsequent requests hit the in-process cache. The endpoint
needs `ROBOFLOW_API_KEY` injected via `environment_vars` from a secret
so the SDK can authenticate the one-time download. No per-frame
requests touch the network.

Payload (matches AppKit `serving()` invoke):

```json
{"dataframe_records": [{"image": "<b64>", "conf": 0.35}]}
```

Response (matches the YOLO endpoint so `_normalizeDatabricks` works):

```json
{"predictions": [[{"label": "...", "class_id": 0, "confidence": 0.8, "bbox": [x1,y1,x2,y2]}]]}
```

In [ ]:
dbutils.widgets.text("catalog", "reggie_pierce_aws_catalog")
dbutils.widgets.text("schema", "lens_iq")
# Per-use-case slug. Used to build both the UC registered model name
# (`lensiq_<slug>`) and the serving endpoint name (`lensiq-<slug-with-dashes>`).
# Match the model ids declared in client/src/lib/models.ts so the AppKit
# server can bind the same alias.
dbutils.widgets.text("model_slug", "license_plate")
dbutils.widgets.text("model_display_name", "License plates")
# Roboflow Universe coordinates. The Universe URL for a project is
#   https://universe.roboflow.com/<workspace>/<project>
# and the SDK model id is `<project>/<version>`. The workspace slug is
# kept around purely for MLflow tagging / lineage; the served PyFunc
# only needs `<project>/<version>`.
dbutils.widgets.text("roboflow_workspace", "samrat-sahoo")
dbutils.widgets.text("roboflow_project", "license-plates-f8vsn")
dbutils.widgets.text("roboflow_version", "5")
# Roboflow API key secret. Injected into the serving endpoint's
# environment_vars so the inference SDK can do its one-time download +
# license check on cold start. Per-frame inference does not use it.
dbutils.widgets.text("api_key_scope", "lens-iq")
dbutils.widgets.text("api_key_secret", "roboflow_api_key")

# Optional output filters applied AFTER local YOLO inference. Public
# Universe models are noisy on out-of-distribution CCTV (false-positive
# bboxes spanning whole frames, single-class projects whose class label
# doesn't match our endpoint name, etc). These filters let us reuse a noisy
# Universe model without retraining by post-processing its predictions:
#
#   - class_label_override: rename every kept prediction's class to this
#     fixed string. Use it to align an upstream class like "sign" or
#     "Water-u6Vi" with the endpoint's slug ("wet_floor_sign", "spill"),
#     keeping the cross-endpoint contract uniform.
#   - min_confidence: drop predictions below this absolute confidence.
#     Acts as a floor on what YOLO returns: the PyFunc passes
#     `max(request_conf, min_confidence)` to `YOLO.predict(conf=...)` so
#     the model never bothers scoring anything below the bar.
#   - min_area_pct / max_area_pct: drop bboxes covering less than (more than)
#     N% of the frame. Reject edge-sliver micro-detections (low) and
#     hallucinated frame-spanning boxes (high).
#   - min_y_center_pct / max_y_center_pct: drop bboxes whose vertical
#     center is above (or below) the given % of the frame. Cones and
#     spills are floor-level; this kills shelf-signage / ceiling false
#     positives (low min) and tight bottom-edge sliver hallucinations
#     (high max) without affecting real detections.
#
# All filters are optional; pass `""` (or 0 / 100 for the bounds) to
# disable. Defaults below are no-ops so the existing detectors keep their
# current behaviour.
dbutils.widgets.text("class_label_override", "")
dbutils.widgets.text("min_confidence", "0.0")
dbutils.widgets.text("min_area_pct", "0.0")
dbutils.widgets.text("max_area_pct", "100.0")
dbutils.widgets.text("min_y_center_pct", "0.0")
dbutils.widgets.text("max_y_center_pct", "100.0")

In [ ]:
%pip install -q mlflow>=2.13
dbutils.library.restartPython()

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("deploy_roboflow_detector")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
MODEL_SLUG = dbutils.widgets.get("model_slug").strip()
MODEL_DISPLAY_NAME = dbutils.widgets.get("model_display_name").strip() or MODEL_SLUG
ROBOFLOW_WORKSPACE = dbutils.widgets.get("roboflow_workspace").strip()
ROBOFLOW_PROJECT = dbutils.widgets.get("roboflow_project").strip()
ROBOFLOW_VERSION = int(dbutils.widgets.get("roboflow_version").strip())
API_KEY_SCOPE = dbutils.widgets.get("api_key_scope")
API_KEY_SECRET = dbutils.widgets.get("api_key_secret")

# Optional post-filter knobs. Empty string / no-op default values mean
# "filter disabled" and reproduce the previous notebook's behaviour
# verbatim for the detectors that don't need filtering (license_plate,
# slip_fall).
CLASS_LABEL_OVERRIDE = dbutils.widgets.get("class_label_override").strip()
MIN_CONFIDENCE = float(dbutils.widgets.get("min_confidence") or 0.0)
MIN_AREA_PCT = float(dbutils.widgets.get("min_area_pct") or 0.0)
MAX_AREA_PCT = float(dbutils.widgets.get("max_area_pct") or 100.0)
MIN_Y_CENTER_PCT = float(dbutils.widgets.get("min_y_center_pct") or 0.0)
MAX_Y_CENTER_PCT = float(dbutils.widgets.get("max_y_center_pct") or 100.0)

# Conventions tying everything together:
#   - UC registered model:  <catalog>.<schema>.lensiq_<slug>
#   - Serving endpoint:     lensiq-<slug-with-dashes>
# The AppKit server expects matching aliases (see server/server.ts).
REGISTERED = f"{CATALOG}.{SCHEMA}.lensiq_{MODEL_SLUG}"
ENDPOINT = f"lensiq-{MODEL_SLUG.replace('_', '-')}"

# Sanity-check the Roboflow key exists. The served PyFunc reads it from
# the endpoint's environment_vars (injected from the same secret), so a
# missing secret means cold start will fail. Catching it here avoids
# logging a broken model version that can never load.
_probe_key = dbutils.secrets.get(scope=API_KEY_SCOPE, key=API_KEY_SECRET)
if not _probe_key:
    raise RuntimeError(
        f"Roboflow API key not found at secrets/{API_KEY_SCOPE}/{API_KEY_SECRET}"
    )

LOG.info("Deploying %s -> %s", MODEL_DISPLAY_NAME, ENDPOINT)
LOG.info("  universe: %s/%s/%s", ROBOFLOW_WORKSPACE, ROBOFLOW_PROJECT, ROBOFLOW_VERSION)
LOG.info("  registered_model=%s", REGISTERED)
LOG.info("  filters: relabel=%r min_conf=%.2f area=[%.2f-%.2f]%% y_center=[%.0f-%.0f]%%",
         CLASS_LABEL_OVERRIDE or None, MIN_CONFIDENCE,
         MIN_AREA_PCT, MAX_AREA_PCT, MIN_Y_CENTER_PCT, MAX_Y_CENTER_PCT)

## Resolve the model id

Just builds the `<project>/<version>` string the SDK expects. No
download happens at deploy time - the PyFunc handles that on cold
start (see `dbx-models/spill_detect_model.py` for the same pattern).
Keeping deploy short avoids two failure modes we hit earlier:
serverless `pip install inference-cpu` taking 10+ min and the
notebook user not having write access to `/local_disk0`.

In [ ]:
MODEL_ID = f"{ROBOFLOW_PROJECT}/{ROBOFLOW_VERSION}"
LOG.info("Will deploy Roboflow Universe model id: %s", MODEL_ID)


## PyFunc wrapper

On cold start, points `MODEL_CACHE_DIR` at a writable temp dir and
calls `inference.get_model(...)` - the SDK downloads weights once,
caches them in-process, and serves all subsequent frames locally.

Important gotchas this wrapper handles:

- **`MODEL_CACHE_DIR` is set BEFORE `from inference import get_model`**.
  The package reads this env var at import time to wire up its internal
  cache layer. Setting it after import is a no-op.
- **Writable temp dir helper** probes `SPARK_LOCAL_DIRS` first then
  `tempfile.gettempdir()` with an actual write probe. `/local_disk0`
  on serverless exists but is read-only for the notebook user. This is
  the same pattern as `dbx-models/mlfow_models.py::tmp_dir()`.
- **API key required at runtime** for the SDK's download +
  license/metadata call. Injected via endpoint `environment_vars` from
  a Databricks secret. Per-frame inference does not touch the network.
- **Telemetry / workflow endpoints suppressed** via env vars so the
  package doesn't ping unrelated Roboflow services.

The post-filter knobs (relabel, min confidence, area, vertical position)
that the proxy PyFunc used to apply to Roboflow's JSON response are
preserved and re-applied to the SDK's prediction list.

In [ ]:
import base64
import io
import os
import tempfile
import uuid

import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.models import infer_signature


def _writable_temp_subdir(name: str) -> str:
    """Return a writable subdirectory matching dbx-models::tmp_dir().

    Probes SPARK_LOCAL_DIRS first (low-latency on serving), falls back
    to tempfile.gettempdir(). Actually writes a probe file so we never
    return a path that looks writable (e.g. /local_disk0) but rejects
    writes from the serving user.
    """
    spark_local_dirs = os.environ.get("SPARK_LOCAL_DIRS")
    spark_local_dir = spark_local_dirs.split(",")[0] if spark_local_dirs else None
    for unique in (False, True):
        suffix = f"_{uuid.uuid4().hex}" if unique else ""
        for base in (spark_local_dir, tempfile.gettempdir()):
            if not base:
                continue
            target = os.path.join(base, f"{name}{suffix}")
            try:
                os.makedirs(target, exist_ok=True)
                probe = os.path.join(target, f".probe_{os.getpid()}")
                with open(probe, "w") as f:
                    f.write("")
                os.remove(probe)
                return target
            except (OSError, PermissionError):
                continue
    raise PermissionError(f"No writable temp dir found for {name}")


class RoboflowDetector(mlflow.pyfunc.PythonModel):
    """PyFunc that runs a Roboflow Universe model locally via the
    `inference` SDK. The SDK downloads weights on the first cold start
    into a writable temp dir, then runs all subsequent inference
    in-process.

    Inputs (per row):
      - image: base64-encoded JPEG/PNG (with or without `data:` prefix).
      - conf:  optional confidence threshold, default 0.35.

    Output (per row): list of `{label, class_id, confidence, bbox}` with
    `bbox` as `[x1, y1, x2, y2]`. Matches the YOLO endpoint's contract.

    Optional output filters (read from `model_config`):
      - class_label_override: rename every kept prediction's class to a
        fixed string. Use when an upstream Universe model's only class
        ("sign", "Water-u6Vi", ...) doesn't match this endpoint's slug.
      - min_confidence: floor on confidence. Passed as the SDK's
        `confidence=` param so the model never bothers scoring boxes
        below it.
      - min_area_pct / max_area_pct: drop bboxes whose area is outside
        this fraction-of-frame range. Kills edge slivers and hallucinated
        full-frame boxes that some out-of-distribution models emit.
      - min_y_center_pct / max_y_center_pct: drop bboxes whose vertical
        center is above (or below) the given % of the frame. Cones and
        spills are floor-level; the lower bound rejects shelf signage /
        ceiling false positives, and the upper bound rejects bottom-edge
        sliver hallucinations some models emit at y=99%.

    These filters live INSIDE the served PyFunc (instead of in the AppKit
    server) so a single notebook + bundle entry can ship a noisy Universe
    model as a clean single-purpose detector. The downstream contract
    that the AppKit server consumes never changes.
    """

    def load_context(self, context):
        # MODEL_CACHE_DIR MUST be set before `from inference import ...`
        # because the package reads it at import time.
        os.environ["MODEL_CACHE_DIR"] = _writable_temp_subdir("rfcache")
        os.environ.setdefault("DISABLE_INFERENCE_TELEMETRY", "True")
        os.environ.setdefault("TELEMETRY_OPT_OUT", "True")
        os.environ.setdefault("DISABLE_WORKFLOW_ENDPOINTS", "True")
        os.environ.setdefault("YOLO_CONFIG_DIR", _writable_temp_subdir("yolo_config"))
        # inference 1.2.0 swapped the default engine to `inference-models`,
        # which returns a different prediction-object shape (no `.predictions`
        # attribute on the result). Our parsing below targets the legacy
        # engine's response. Force the legacy engine until we adopt the new
        # shape (and to keep the deployed contract stable across SDK bumps).
        os.environ["USE_INFERENCE_MODELS"] = "False"

        from inference import get_model
        from PIL import Image

        self._Image = Image

        cfg = context.model_config or {}
        model_id = cfg["model_id"]
        self._class_label_override = (cfg.get("class_label_override") or "").strip() or None
        self._min_confidence = float(cfg.get("min_confidence") or 0.0)
        self._min_area_pct = float(cfg.get("min_area_pct") or 0.0)
        self._max_area_pct = float(cfg.get("max_area_pct") or 100.0)
        self._min_y_center_pct = float(cfg.get("min_y_center_pct") or 0.0)
        self._max_y_center_pct = float(cfg.get("max_y_center_pct") or 100.0)

        api_key = os.environ.get("ROBOFLOW_API_KEY", "")
        if not api_key:
            raise RuntimeError(
                "ROBOFLOW_API_KEY env var not set on serving endpoint. "
                "Inject it via ServedEntityInput.environment_vars."
            )
        self._model = get_model(model_id=model_id, api_key=api_key)

    def _decode(self, image_b64):
        if not image_b64:
            return None
        if isinstance(image_b64, str) and image_b64.startswith("data:"):
            image_b64 = image_b64.split(",", 1)[1]
        return self._Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")

    def _keep(self, x1, y1, x2, y2, confidence, image_w, image_h):
        """Apply geometric + confidence post-filters. Returns True if the
        box survives. Confidence is enforced HERE (not via the SDK's
        `confidence=` param) because the inference SDK applies a model-
        and backend-specific score recalibration that diverges from the
        Roboflow hosted API's reported confidences, sometimes dropping
        every prediction even when the hosted API says they all clear
        the threshold."""
        if confidence < self._min_confidence:
            return False
        if not image_w or not image_h:
            return True
        w = max(0, x2 - x1)
        h = max(0, y2 - y1)
        area_pct = (w * h) / (image_w * image_h) * 100.0
        if area_pct < self._min_area_pct or area_pct > self._max_area_pct:
            return False
        y_center_pct = ((y1 + y2) / 2.0) / image_h * 100.0
        if y_center_pct < self._min_y_center_pct:
            return False
        if y_center_pct > self._max_y_center_pct:
            return False
        return True

    # SDK confidence floor. Always send a very permissive threshold to the
    # SDK and enforce min_confidence ourselves in _keep, because the SDK's
    # local score post-processing diverges from the Roboflow hosted API
    # and over-prunes (see comment in _keep). 1% catches the long tail
    # without exploding response size.
    _SDK_CONFIDENCE_FLOOR = 0.01

    def _run_one(self, image_b64, conf):
        img = self._decode(image_b64)
        if img is None:
            return []
        import numpy as np

        arr = np.array(img)
        image_h, image_w = arr.shape[:2]
        # The SDK's infer() accepts a numpy array, PIL Image, or file
        # path. It returns a list of ObjectDetectionInferenceResponse,
        # one per input image; each has `.predictions` with center-xywh
        # bboxes in pixel coords.
        results = self._model.infer(arr, confidence=self._SDK_CONFIDENCE_FLOOR)
        if not isinstance(results, list):
            results = [results]
        out = []
        for r in results:
            for p in getattr(r, "predictions", []) or []:
                cx = float(getattr(p, "x", 0.0))
                cy = float(getattr(p, "y", 0.0))
                w = float(getattr(p, "width", 0.0))
                h = float(getattr(p, "height", 0.0))
                x1 = int(round(cx - w / 2))
                y1 = int(round(cy - h / 2))
                x2 = int(round(cx + w / 2))
                y2 = int(round(cy + h / 2))
                cls_conf = float(getattr(p, "confidence", 0.0))
                # Honor the caller's request floor too: many client calls
                # pass conf=0.35 as a "show me serious detections" hint.
                requester_floor = float(conf or 0.0)
                if cls_conf < requester_floor:
                    continue
                if not self._keep(x1, y1, x2, y2, cls_conf, image_w, image_h):
                    continue
                label = self._class_label_override or getattr(p, "class_name", None) or "object"
                out.append({
                    "label": label,
                    "class_id": int(getattr(p, "class_id", -1)),
                    "confidence": cls_conf,
                    "bbox": [x1, y1, x2, y2],
                })
        return out

    def predict(self, context, model_input, params=None):
        if hasattr(model_input, "to_dict"):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)
        return [self._run_one(r.get("image"), r.get("conf")) for r in rows]

## Log + register

In [ ]:
_TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAH"
    "ggJ/PchI7wAAAABJRU5ErkJggg=="
)

sample_input = pd.DataFrame([
    {"image": _TINY_PNG_B64, "conf": 0.35},
])
sample_output = [[]]
signature = infer_signature(sample_input, sample_output)

mlflow.set_registry_uri("databricks-uc")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# model_config carries the model id (so the served PyFunc knows what to
# load via get_model) and the static post-filter knobs. The populated
# cache travels via `artifacts`. The Roboflow API key is intentionally
# NOT in the artifact - it's injected at serve time from a secret.
model_config = {
    "model_id": MODEL_ID,
    "class_label_override": CLASS_LABEL_OVERRIDE,
    "min_confidence": MIN_CONFIDENCE,
    "min_area_pct": MIN_AREA_PCT,
    "max_area_pct": MAX_AREA_PCT,
    "min_y_center_pct": MIN_Y_CENTER_PCT,
    "max_y_center_pct": MAX_Y_CENTER_PCT,
}

with mlflow.start_run(run_name=f"deploy_roboflow_{MODEL_SLUG}") as run:
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=RoboflowDetector(),
        signature=signature,
        input_example=sample_input,
        registered_model_name=REGISTERED,
        model_config=model_config,
        pip_requirements=[
            "mlflow>=2.13",
            # Pin below 1.2.0: starting at inference-cpu 1.2.0 the default
            # backend switched to `inference-models`, whose `infer()` result
            # is a list of detection objects without the legacy
            # `.predictions` attribute that our PyFunc parses. Setting
            # `USE_INFERENCE_MODELS=False` is not reliable because the
            # package imports occur before the env var is read in some code
            # paths. Pinning is the durable fix until we adopt the new
            # response shape.
            "inference-cpu>=1.1.0,<1.2.0",
        ],
    )
    mlflow.set_tag("lensiq.detector_slug", MODEL_SLUG)
    mlflow.set_tag("lensiq.display_name", MODEL_DISPLAY_NAME)
    mlflow.set_tag("lensiq.roboflow_workspace", ROBOFLOW_WORKSPACE)
    mlflow.set_tag("lensiq.roboflow_project", ROBOFLOW_PROJECT)
    mlflow.set_tag("lensiq.roboflow_version", str(ROBOFLOW_VERSION))
    if CLASS_LABEL_OVERRIDE:
        mlflow.set_tag("lensiq.class_label_override", CLASS_LABEL_OVERRIDE)
LOG.info("Logged model URI: %s", info.model_uri)

## Create / update the serving endpoint

Pulls `ROBOFLOW_API_KEY` from the configured Databricks secret via
`environment_vars` so the inference SDK can do its one-time
license/metadata call on cold start. After load, all inference is
local; the secret is never used for per-frame requests.

First-time endpoint creation builds a container image with
`inference-cpu` (~500MB including onnxruntime + opencv + fastapi),
which takes ~8-12 minutes. Subsequent config updates redeploy the
existing image and only take ~1-2 minutes.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED}'")
latest_version = max(versions, key=lambda v: int(v.version)).version
LOG.info("Deploying %s version %s -> endpoint %s", REGISTERED, latest_version, ENDPOINT)

served = ServedEntityInput(
    entity_name=REGISTERED,
    entity_version=latest_version,
    workload_size="Small",
    scale_to_zero_enabled=True,
    environment_vars={
        # Required by the inference SDK on cold start for license /
        # metadata validation. Per-frame inference does not use it.
        "ROBOFLOW_API_KEY": f"{{{{secrets/{API_KEY_SCOPE}/{API_KEY_SECRET}}}}}",
        # Suppress telemetry pings and unrelated workflow endpoints.
        "DISABLE_INFERENCE_TELEMETRY": "True",
        "TELEMETRY_OPT_OUT": "True",
        "DISABLE_WORKFLOW_ENDPOINTS": "True",
        # inference >=1.2.0 defaults to the new `inference-models` engine
        # whose `infer()` response is a list of detection objects without
        # the legacy `.predictions` attribute our PyFunc parses. Pin the
        # legacy backend so the deployed contract stays stable across SDK
        # bumps (also set in load_context for belt-and-suspenders).
        "USE_INFERENCE_MODELS": "False",
    },
)

w = WorkspaceClient()
try:
    w.serving_endpoints.get(name=ENDPOINT)
    LOG.info("Endpoint exists; updating config")
    w.serving_endpoints.update_config(name=ENDPOINT, served_entities=[served])
except Exception:
    LOG.info("Endpoint not found; creating")
    w.serving_endpoints.create(
        name=ENDPOINT,
        config=EndpointCoreConfigInput(name=ENDPOINT, served_entities=[served]),
    )
LOG.info("Submitted deployment for %s; watch Serving UI for readiness.", ENDPOINT)